# FAO DATALAB

Objectif : produire une analyse détaillée sur les données de la FAO de 2023 sur la sous-nutrition à l'échelle mondiale

## Intro. Initialisation du projet

**A. Importation des librairies**

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
sns.set_theme(style="whitegrid")

**B. Importation des datasets**

*Import de 5 fichiers CSV de FAOSTAT tous structurés selon le schéma standard FAO : Code Domaine, Domaine, Code zone, Zone, Code Élément, Élément, Code Produit, Produit, Code année, Année, Unité, Valeur, Note*

In [2]:
df_animaux = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_animaux.csv")
df_cereales = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_céréales.csv")
df_population = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_population.csv")
df_vegetaux = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_vegetaux.csv")
df_sousalimentation = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\Périodes_glissantes\fr_sousalimentation.csv")

**C. Vérification de l'importation - Encodage**

In [3]:
df_animaux.head()
df_cereales.head()
df_population.head()
df_vegetaux.head()
df_sousalimentation.head()

,Code Domaine,Domaine,Code zone (FAO),Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20192021,2019-2021,%,28.9,E,Valeur estimée,NaN
1,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20202022,2020-2022,%,31,E,Valeur estimée,NaN
2,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20212023,2021-2023,%,32,E,Valeur estimée,NaN
3,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20222024,2022-2024,%,29.7,E,Valeur estimée,NaN
4,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20232025,2023-2025,%,27.8,E,Valeur estimée,NaN


**D. Visualisation des dimensions**

*La visualisation est faite pour rendre compte de la structure de chaque fichier*

In [4]:
print(f"vegetaux          | {df_vegetaux.shape[0]:6} lignes | {df_vegetaux.shape[1]} colonnes")
print(f"animaux           | {df_animaux.shape[0]:6} lignes | {df_animaux.shape[1]} colonnes")
print(f"cereales          | {df_cereales.shape[0]:6} lignes | {df_cereales.shape[1]} colonnes")
print(f"population        | {df_population.shape[0]:6} lignes | {df_population.shape[1]} colonnes")
print(f"sousalimentation  | {df_sousalimentation.shape[0]:6} lignes | {df_sousalimentation.shape[1]} colonnes")

vegetaux          |  91617 lignes | 15 colonnes
animaux           |  33432 lignes | 15 colonnes
cereales          |  14203 lignes | 15 colonnes
population        |    177 lignes | 15 colonnes
sousalimentation  |   2040 lignes | 15 colonnes


**Constat :** les 5 datasets de la FAO issus de FAOSTATS pour l'année 2023 sont formatés de la même manière : les 15 colonnes du schéma standard susmentionné

# Etape 1 : Diagnostic qualité et préparation des données

## 1. Diagnostic : Audit technique et POO
*Présentation de la classe 'DataProfiler' et 'ProfileurFAO'*

**A. Classe Mère**

*Il s'agit d'une classe générique permettant qu'être utilisée pour tous autres fichiers indépendemment du sujet traité dans ce projet*


In [5]:
class DataProfiler:
    """
    Classe mère générique de diagnostic qualité pour tout type de dataset.
    Permet de centraliser les contrôles structurels de base (taille, NaN, doublons).
    """
    
    def __init__(self, df: pd.DataFrame, nom_source: str):
        self.df = df
        self.nom_source = nom_source

    def rapport_nan_global(self) -> pd.DataFrame:
        """
        Calcul du nombre et du pourcentage de valeurs manquantes pour chaque colonne.
        Paramètres : Aucun
        Retourne : pd.Dataframe -> Un tableau synthétique listant les colonnes, le nb de NaN et leur taux en %.
        """
        total_lignes = len(self.df)
        nan_count = self.df.isna().sum()
        nan_pct = (nan_count / total_lignes * 100) if total_lignes > 0 else nan_count * 0
        
        df_synthese = pd.DataFrame({
            "Colonne": nan_count.index,
            "Nb_NaN": nan_count.values,
            "Taux_Pct": nan_pct.values.round(2)
        })
        return df_synthese[df_synthese["Nb_NaN"] > 0].reset_index(drop=True)

    def rapport_doublons(self, subset=None) -> int:
        """
        Identification et comptage du nombre de lignes dupliquées.
        Paramètres : subset (list ou str, optionnel) -> Colonnes à considérer pour la recherche de doublons.
        Retourne : int -> Le nombre total de doublons stricts ou sur le subset choisi.
        """
        return int(self.df.duplicated(subset=subset).sum())

    def detecter_valeurs_atypiques(self, colonne: str, seuil_min=None, seuil_max=None) -> pd.DataFrame:
        """
        Isolement des lignes sortant de plages plausibles métier (ex: valeurs aberrantes de disponibilité).
        Paramètres :
        colonne : str -> Nom de la colonne à auditer.
        seuil_min : float, optionnel -> Seuil minimal acceptable.
        seuil_max : float, optionnel -> Seuil maximal acceptable.
        Retourne : pd.DataFrame -> Sous-ensemble du DataFrame contenant uniquement les lignes atypiques.
        """
        if colonne not in self.df.columns:
            return pd.DataFrame()
        
        s=pd.to_numeric(self.df[colonne], errors='coerce')

        masque = pd.Series([False] * len(self.df), index=self.df.index)
        if seuil_min is not None:
            masque = masque | (s < seuil_min)
        if seuil_max is not None:
            masque = masque | (s > seuil_max)

        return self.df[masque]
    
    def zones_uniques(self) -> set:
        """
        Retourne l'ensemble des zones (pays) uniques présentes dans la source.
        Paramètres : Aucun
        Retourne : set -> Ensemble des zones géographiques.
        """
        if "Zone" in self.df.columns:
            return set(self.df["Zone"].dropna().unique())
        return set()

    def resume_audit(self) -> dict:
        """
        Génèration d'un dictionnaire de synthèse globale de la source.
        Évite les prints bruts pour faciliter l'intégration dans un tableau de bord ou un DataFrame.
        Paramètres : Aucun
        Retourne : dict -> Dictionnaire contenant le nom de la source, le nombre de lignes et de colonnes.
        """
        return {
            "Source": self.nom_source,
            "Nb_Lignes": len(self.df),
            "Nb_Colonnes": len(self.df.columns)
        }


**B. Classe Fille**

*Classe spécifique à notre dataset qui va aller chercher les spécificités de nos jeux de données*

In [6]:
class ProfileurFAO(DataProfiler):
    """
    Classe fille spécialisée pour auditer les spécificités des fichiers FAO 
    (gestion des clés de jointure géographiques et des seuils nutritionnels).
    """
    
    CLE_JOINTURE = "Zone"

    def verifier_coherence_cles(self, autre_df: pd.DataFrame) -> set:
        """
        Identification des pays présents dans la table courante mais absents d'une autre table de référence.
        Paramètres : autre_df : pd.DataFrame -> Le DataFrame avec lequel comparer les zones géographiques.
        Retourne : set -> L'ensemble des zones (pays) en rupture de jointure.
        """
        zones_source = set(self.df[self.CLE_JOINTURE].dropna())
        zones_autre = set(autre_df[self.CLE_JOINTURE].dropna())
        return zones_source - zones_autre

    def recenser_unites(self) -> pd.DataFrame:
        """
        Recensemment des couples uniques entre les libellés d'Élément et leurs Unités de mesure.
        Paramètres : Aucun    
        Retourne : pd.DataFrame -> Un tableau listant les combinaisons Élément / Unité présentes dans le dataset.
        """
        if 'Élément' in self.df.columns and 'Unité' in self.df.columns:
            return self.df[['Élément', 'Unité']].drop_duplicates().reset_index(drop=True)
        return pd.DataFrame()

    # Dictionnaire de correspondance exhaustif des symboles FAO
    DICT_SYMBOLES_FAO = {
        'A': 'Donnée officielle (Official)',
        'E': 'Donnée estimée (Estimated)',
        'I': 'Donnée imputée par la FAO (Imputed)',
        'M': 'Donnée manquante (Missing)',
        'O': 'Valeur manquante (Missing value)',
        'Q': 'Valeur manquante ou masquée (Missing/Suppressed)',
        'S': 'Donnée non officielle / agrégée',
        'X': 'Source externe / Organisation internationale',
        '*': 'Donnée non officielle',
        'Fc': 'Donnée calculée (Calculated)'
    }

    def analyser_fiabilite_symboles(self) -> pd.DataFrame:
        """
        Analyse de la répartition statistique de la colonne Symbole pour évaluer la fiabilité des données.
        Paramètres : Aucun  
        Retourne : pd.DataFrame -> Un tableau croisé indiquant l'effectif et le pourcentage de chaque symbole de fiabilité.
        """
        if 'Symbole' in self.df.columns:
            total = len(self.df)
            counts = self.df['Symbole'].value_counts()
            pct = (counts / total * 100).round(2)

            df_sym = pd.DataFrame({"Effectif": counts, "Part_Pct": pct}).reset_index()
            df_sym.columns = ["Symbole", "Effectif", "Part_Pct"]
            
            # Ajout de la colonne explicative
            df_sym["Signification"] = df_sym["Symbole"].map(self.DICT_SYMBOLES_FAO).fillna("Donnée officielle / Non renseigné")
            
            return df_sym
        return pd.DataFrame()

**C. Exécution des méthodes de profiling sur les 5 fichiers sources**

*Objectif : Appliquer nos méthodes sur l'ensemble des CSV afin d'observer les spécificités, les variations ou encore les données "suspectes"*

In [7]:
# STOCKAGE des 5 DataFrames et leurs noms dans un dictionnaire pour les parcourir facilement
fichiers_dict = {
    "Animaux": df_animaux,
    "Céréales": df_cereales,
    "Population": df_population,
    "Végétaux": df_vegetaux,
    "Sous-alimentation": df_sousalimentation
}

print("##################################################")
print("### PARTIE A : CONTRÔLES STRUCTURELS           ###")
print("##################################################\n")

# --- 1. TABLEAU DE SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ---
synthese_globale = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    synthese_globale.append(profiler.resume_audit())

df_synthese_globale = pd.DataFrame(synthese_globale)
print("\n=== 1. SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ===")
display(df_synthese_globale)

# --- 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ---
synthese_nan = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    df_nan = profiler.rapport_nan_global()
    # On filtre uniquement les colonnes qui ont des NaN pour ne garder que l'essentiel
    df_nan_filtré = df_nan[df_nan["Nb_NaN"] > 0].copy()
    if not df_nan_filtré.empty:
        df_nan_filtré["Source"] = nom
        synthese_nan.append(df_nan_filtré)

if synthese_nan:
    df_synthese_nan = pd.concat(synthese_nan, ignore_index=True)
    df_synthese_nan = df_synthese_nan[["Source", "Colonne", "Nb_NaN", "Taux_Pct"]]
    print("=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===")
    display(df_synthese_nan)
else:
    print("=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===")
    print("Aucune valeur manquante détectée dans les fichiers !")


# --- 3. RAPPORT DES DOUBLONS ---
synthese_doublons = []
for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    
    # Doublons stricts (sur toute la ligne)
    doublons_stricts = profiler.rapport_doublons(subset=None)
    
    # Doublons sur clé métier (Zone, Élément, Produit si disponible)
    colonnes_dispo = [col for col in ['Zone', 'Élément', 'Produit'] if col in df.columns]
    doublons_metier = profiler.rapport_doublons(subset=colonnes_dispo) if len(colonnes_dispo) > 1 else 0

    synthese_doublons.append({
        "Source": nom,
        "Doublons_Stricts": doublons_stricts,
        "Doublons_Cle_Metier": doublons_metier
    })

df_synthese_doublons = pd.DataFrame(synthese_doublons)
print("\n=== 3. RAPPORT DES DOUBLONS (Stricts et Clé Métier) ===")
display(df_synthese_doublons)

pd.set_option('display.max_colwidth', None)
# --- 4. VALEURS NÉGATIVES / ATYPIQUES (Détail par Colonne et Élément) ---
print("\n=== 4. RAPPORT DES VALEURS NÉGATIVES / ATYPIQUES ===")

synthese_atypiques = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    
    # Isolation des lignes où Valeur < 0
    df_neg = profiler.detecter_valeurs_atypiques(colonne='Valeur', seuil_min=0)
    
    if not df_neg.empty:
        # Récupération de la valeur minimale
        val_min = pd.to_numeric(df_neg['Valeur'], errors='coerce').min()
        
        # Décompte et formatage des éléments impactés (ex: "Variation de stock (200), Disponibilité intérieure (109)")
        if 'Élément' in df_neg.columns:
            counts = df_neg['Élément'].value_counts()
            elements_str = ", ".join([f"{elt} ({count})" for elt, count in counts.items()])
        else:
            elements_str = "Non spécifié"
            
        synthese_atypiques.append({
            "Source": nom,
            "Colonne": "Valeur",
            "Nb_Valeurs_Négatives": len(df_neg),
            "Valeur_Min": val_min,
            "Répartition_Éléments": elements_str
        })
    else:
        synthese_atypiques.append({
            "Source": nom,
            "Colonne": "Valeur",
            "Nb_Valeurs_Négatives": 0,
            "Valeur_Min": "-",
            "Répartition_Éléments": "Aucune"
        })

df_synthese_atypiques = pd.DataFrame(synthese_atypiques)
display(df_synthese_atypiques)

##################################################
### PARTIE A : CONTRÔLES STRUCTURELS           ###
##################################################


=== 1. SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ===


,Source,Nb_Lignes,Nb_Colonnes
0,Animaux,33432,15
1,Céréales,14203,15
2,Population,177,15
3,Végétaux,91617,15
4,Sous-alimentation,2040,15


=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===


,Source,Colonne,Nb_NaN,Taux_Pct
0,Animaux,Note,33432,100.00
1,Céréales,Note,14203,100.00
2,Population,Note,177,100.00
3,Végétaux,Note,91617,100.00
4,Sous-alimentation,Valeur,668,32.75
5,Sous-alimentation,Note,2040,100.00



=== 3. RAPPORT DES DOUBLONS (Stricts et Clé Métier) ===


,Source,Doublons_Stricts,Doublons_Cle_Metier
0,Animaux,0,0
1,Céréales,0,0
2,Population,0,0
3,Végétaux,0,0
4,Sous-alimentation,0,1632



=== 4. RAPPORT DES VALEURS NÉGATIVES / ATYPIQUES ===


,Source,Colonne,Nb_Valeurs_Négatives,Valeur_Min,Répartition_Éléments
0,Animaux,Valeur,129,-1107.0,"Variation de stock (109), Disponibilité intérieure (20)"
1,Céréales,Valeur,309,-5530.0,"Variation de stock (291), Disponibilité intérieure (18)"
2,Population,Valeur,0,-,Aucune
3,Végétaux,Valeur,243,-1907.0,Disponibilité intérieure (243)
4,Sous-alimentation,Valeur,0,-,Aucune


**Constat :** Les 5 fichiers présentent une structure cohérente et standardisée de 15 colonnes par table. La volumétrie globale est très variable (de 177 lignes pour Population à 91 617 lignes pour Végétaux), ce qui reflète la diversité du niveau de détail entre les bilans de masse volumineux et les données démographiques agrégées.

Valeurs Manquantes : La colonne "Note" présente un taux de vacuité absolu de 100 % sur l'intégralité des 5 fichiers. Il s'agit d'une colonne système résiduelle inutilisée lors de l'export FAO, qui pourra être purement et simplement supprimée lors du nettoyage.
La colonne Valeur (Sous-alimentation) a quant à elle 668 valeurs sont manquantes, soit 32,75 % de la table. Cette vacuité n'est pas un défaut d'extraction technique : la FAO ne publie pas de chiffres de sous-alimentation pour les très petits territoires ou lorsque la prévalence est estimée en dessous du seuil de détection statistique (souvent noté <2,5 % ou omis dans les publications officielles).

Doublons stricts : Aucun doublon parfait n'est détecté sur l'ensemble de la base (0 sur toutes les tables), ce qui garantit l'intégrité technique de l'importation.
Doublons sur clé métier (Zone, Élément, Produit) : Parfaitement nuls sur les bilans alimentaires et la population. En revanche, 1 632 doublons de clé apparaissent dans la table Sous-alimentation. Cela est tout à fait normal : cette table suit une logique temporelle par périodes triennales glissantes (ex: 2012-2014, 2013-2015) et contient plusieurs indicateurs. Sans l'intégration de la colonne Année/Période dans la clé primaire, ces lignes apparaissent logiquement comme répétitives.

L'audit révèle 681 valeurs négatives concentrées exclusivement sur les trois fichiers de bilans alimentaires (309 dans Céréales, 243 dans Végétaux et 129 dans Animaux, avec un minimum extrême à -5 530), tandis que Population et Sous-alimentation en sont exempts. Sur le plan métier, ces données se divisent en deux catégories : 400 lignes de « Variation de stock » négatives (majoritaires dans Céréales et Animaux), qui sont parfaitement légitimes et traduisent un déstockage net sur la période, et 281 lignes de « Disponibilité intérieure » négatives (100 % des cas de Végétaux, ainsi que quelques lignes dans Animaux et Céréales), qui découlent d'un déséquilibre déclaratif dans l'équation du bilan de masse FAO. En phase de nettoyage, les variations de stock pourront être conservées intactes pour le calcul des ressources, alors que les disponibilités intérieures négatives devront faire l'objet d'un retraitement spécifique (recalcul ou ajustement à zéro).




In [8]:
print("##################################################")
print("### PARTIE B : SPÉCIFICITÉS FAO                ###")
print("##################################################\n")

# --- ANALYSE SPECIFIQUE : NaN par période (sous-alimentation) ---
nan_sousalim = df_sousalimentation[df_sousalimentation['Valeur'].isna()]
colonne_periode = 'Année' if 'Année' in df_sousalimentation.columns else 'Période'
repartition_nan_periode = nan_sousalim[colonne_periode].value_counts().reset_index()
repartition_nan_periode.columns = [colonne_periode, 'Nombre_NaN']

print(f"\n===  ANALYSE SPECIFIQUE :  NaN PAR PÉRIODE (Sous-alimentation) ===")
print(f"Total des valeurs manquantes : {len(nan_sousalim)}")
display(repartition_nan_periode)

# --- COUVERTURE GÉOGRAPHIQUE CROISÉE ENTRE TOUS LES FICHIERS (via zones_uniques et cohérence) ---
print("\n=== COUVERTURE GÉOGRAPHIQUE CROISÉE (Nombre de zones uniques par fichier) ===")
# Ensemble complet de tous les pays uniques (tous fichiers confondus)
tous_les_pays = set().union(*[ProfileurFAO(df, nom).zones_uniques() for nom, df in fichiers_dict.items()])

# Tableau de synthèse des écarts
synthese_ecarts = []
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    zones = profiler.zones_uniques()
    manquants = sorted(list(tous_les_pays - zones))
    
    synthese_ecarts.append({
        "Source": nom,
        "Nb_Zones": len(zones),
        "Nb_Manquants": len(manquants),
    })

zones_sousalim = ProfileurFAO(df_sousalimentation, "Sous-alimentation").zones_uniques()


fichiers_autres = [df_animaux, df_cereales, df_population, df_vegetaux]
zones_autres = set().union(*[ProfileurFAO(df, "").zones_uniques() for df in fichiers_autres])
pays_exclusifs = zones_sousalim - zones_autres
print(f"{len(pays_exclusifs)} pays présents dans sousalimentation mais absents des autres fichiers :\n")
for pays in sorted(pays_exclusifs):
    print(f" - {pays}")

display(pd.DataFrame(synthese_ecarts))

# --- RECENSSEMENT DES UNITÉS DE MESURE PAR FICHIER ---
print("\n=== RECENSSEMENT DES UNITÉS DE MESURE ===")
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    df_unites = profiler.recenser_unites()
    if not df_unites.empty:
        print(f"\n--- Unités pour la source : {nom} ---")
        display(df_unites)


# --- FIABILITÉ DE LA DONNÉE (Répartition des Symboles) ---
print("\n=== FIABILITÉ DE LA DONNÉE (Répartition de la colonne Symbole) ===")
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    df_sym = profiler.analyser_fiabilite_symboles()
    if not df_sym.empty:
        print(f"\n--- Symboles pour la source : {nom} ---")
        display(df_sym)

##################################################
### PARTIE B : SPÉCIFICITÉS FAO                ###
##################################################


===  ANALYSE SPECIFIQUE :  NaN PAR PÉRIODE (Sous-alimentation) ===
Total des valeurs manquantes : 668


,Année,Nombre_NaN
0,2023-2025,139
1,2022-2024,136
2,2021-2023,134
3,2020-2022,133
4,2019-2021,126



=== COUVERTURE GÉOGRAPHIQUE CROISÉE (Nombre de zones uniques par fichier) ===
27 pays présents dans sousalimentation mais absents des autres fichiers :

 - Andorre
 - Bermudes
 - Brunéi Darussalam
 - Burundi
 - Bénin
 - Cuba
 - Dominique
 - Groenland
 - Guinée équatoriale
 - Japon
 - Mali
 - Nioué
 - Palaos
 - Palestine
 - Porto Rico
 - République centrafricaine
 - République populaire démocratique de Corée
 - Samoa américaines
 - Singapour
 - Somalie
 - Soudan
 - Soudan du Sud
 - Tchad
 - Togo
 - Tokélaou
 - Érythrée
 - Îles Cook


,Source,Nb_Zones,Nb_Manquants
0,Animaux,177,27
1,Céréales,177,27
2,Population,177,27
3,Végétaux,177,27
4,Sous-alimentation,204,0



=== RECENSSEMENT DES UNITÉS DE MESURE ===

--- Unités pour la source : Animaux ---


...

,Symbole,Effectif,Part_Pct,Signification
0,X,177,100.0,Source externe / Organisation internationale



--- Symboles pour la source : Végétaux ---


,Symbole,Effectif,Part_Pct,Signification
0,I,56516,61.69,Donnée imputée par la FAO (Imputed)
1,E,35101,38.31,Donnée estimée (Estimated)



--- Symboles pour la source : Sous-alimentation ---


,Symbole,Effectif,Part_Pct,Signification
0,E,1372,67.25,Donnée estimée (Estimated)
1,O,360,17.65,Valeur manquante (Missing value)
2,Q,308,15.10,Valeur manquante ou masquée (Missing/Suppressed)


**Constat spécifique aux données de la FAO** : 

Couverture géographique : L'analyse croisée révèle un socle commun de 177 pays sur les 4 fichiers principaux (Animaux, Céréales, Végétaux, Population), tandis que la table Sous-alimentation s'étend sur 204 zones (soit 27 pays exclusifs tels que le Japon, Singapour ou Andorre). Les 668 valeurs manquantes identifiées dans la sous-alimentation (réparties de manière stable à environ 130 NaNs par période) s'expliquent par cette différence de champ géographique et par des restrictions de diffusion de la FAO sur certaines zones. On note également la possibilité de manque de registres douaniers et agricoles exploitatbles pour bâtir des bilans de masse du fait de confits, d'isolement politiques ou de statuts de micro-territoires. Ce décalage n'est pas une anomalie de données, mais une limite méthodologique d'origine : les jointures d'analyse devront se restreindre à l'intersection des 177 pays pour garantir des bilans cohérents.

Harmonisation des unités de mesure : Les unités de mesure sont parfaitement standardisées par type d'indicateur (1000 t pour les flux physiques, 1000 No pour les effectifs de population, kcal/personne/jour et g/personne/jour pour les apports nutritionnels). Il n'y a pas d'incohérence de libellé au sein des tables, mais leur présence sous des ordres de grandeur distincts impose une règle de gestion stricte lors du nettoyage : des conversions systématiques seront obligatoires avant de croiser les tables pour calculer des ratios mondiaux exacts.

Qualité et fiabilité des données : La table Population repose à 100 % sur des données externes (symbole X), tandis que les trois fichiers de bilans alimentaires ne contiennent aucune donnée officielle brute, se partageant uniquement entre données estimées (E) et données imputées par la FAO (I).ette bascule entre estimation et imputation établit un gradient de confiance : les bilans végétaux et céréaliers intègrent une incertitude statistique plus élevée que les bilans animaux. Cette variabilité devra être prise en compte pour nuancer les analyses sur les pays dont les données sont très fortement imputées.

## 2. Rapport d'anomalies quantifié

**Présentation synthétique et chiffré des anomalies détectées permettant une aide à la prise de décisions stratégiques en lien avec le contexte du projet**

In [9]:
import pandas as pd
from IPython.display import display, HTML

# --- RAPPORT D'ANOMALIE QUANTIFIÉ ET SYNTHÈSE DIAGNOSTIC ---
synthese = pd.DataFrame([
    {
        "Point de diagnostic": "Valeurs manquantes",
        "Résultat": "Données absentes de la colonne note dans tous les fichiers (à supprimer) + 668 NaN uniquement dans Sous-alimentation (~130 par période, 0 dans les 4 autres fichiers)",
        "Nature": "Structurel FAO — Incomplétude liée aux zones de crise, régimes fermés et masquage de données"
    },
    {
        "Point de diagnostic": "Doublons",
        "Résultat": "0 doublon détecté (ligne complète et clé métier), sur l'ensemble des 5 fichiers",
        "Nature": "Aucune action requise"
    },
    {
        "Point de diagnostic": "Valeurs négatives / extrêmes",
        "Résultat": "681 valeurs négatives (309 Céréales, 243 Végétaux, 129 Animaux) : 400 'Variation de stock', 281 'Disponibilité intérieure'",
        "Nature": "400 déstockages légitimes (à conserver) ; 281 anomalies comptables déclaratives (à recalculer ou imputer à 0)"
    },
    {
        "Point de diagnostic": "Couverture géographique",
        "Résultat": "177 pays (Veg/Ani/Cer/Pop) vs 204 (Sous-alimentation) -> 27 pays perdus à la jointure",
        "Nature": "Limite méthodologique (micro-territoires & régimes fermés) — Choix de jointure restreinte au socle des 177 pays"
    },
    {
        "Point de diagnostic": "Unités de mesure",
        "Résultat": "5 unités mélangées ('1000 t', '1000 No', 'kcal/personne/jour', 'g/personne/jour', 'kg/personne/an')",
        "Nature": "Point de vigilance technique — Conversions explicites obligatoires (x1 000 pop, x1 000 000 kg) avant aggregation"
    },
    {
        "Point de diagnostic": "Fiabilité (Symbole)",
        "Résultat": "0% de données brutes officielles : 100% X (Pop), ~61.5% I (Végétaux/Céréales), 61.8% E (Animaux)",
        "Nature": "Axe de lecture qualité — Gradient de confiance (plus forte incertitude sur le végétal/céréales) à conserver"
    },
    {
        "Point de diagnostic": "Particularité métier (Formatage)",
        "Résultat": "Présence de la notation textuelle '<0.1' (ou valeurs seuils) dans la colonne Valeur de Sous-alimentation",
        "Nature": "Spécificité métier FAO — Impossibilité de conversion float directe ; décision de traitement : imputation à 0.05 ou 0.1"
    }
])


pd.set_option('display.max_colwidth', None)

# Affichage sous forme de tableau propre
display(HTML(synthese.to_html(index=False)))

Point de diagnostic,Résultat,Nature
Valeurs manquantes,"Données absentes de la colonne note dans tous les fichiers (à supprimer) + 668 NaN uniquement dans Sous-alimentation (~130 par période, 0 dans les 4 autres fichiers)","Structurel FAO — Incomplétude liée aux zones de crise, régimes fermés et masquage de données"
Doublons,"0 doublon détecté (ligne complète et clé métier), sur l'ensemble des 5 fichiers",Aucune action requise
Valeurs négatives / extrêmes,"681 valeurs négatives (309 Céréales, 243 Végétaux, 129 Animaux) : 400 'Variation de stock', 281 'Disponibilité intérieure'",400 déstockages légitimes (à conserver) ; 281 anomalies comptables déclaratives (à recalculer ou imputer à 0)
Couverture géographique,177 pays (Veg/Ani/Cer/Pop) vs 204 (Sous-alimentation) -> 27 pays perdus à la jointure,Limite méthodologique (micro-territoires & régimes fermés) — Choix de jointure restreinte au socle des 177 pays
Unités de mesure,"5 unités mélangées ('1000 t', '1000 No', 'kcal/personne/jour', 'g/personne/jour', 'kg/personne/an')","Point de vigilance technique — Conversions explicites obligatoires (x1 000 pop, x1 000 000 kg) avant aggregation"
Fiabilité (Symbole),"0% de données brutes officielles : 100% X (Pop), ~61.5% I (Végétaux/Céréales), 61.8% E (Animaux)",Axe de lecture qualité — Gradient de confiance (plus forte incertitude sur le végétal/céréales) à conserver
Particularité métier (Formatage),Présence de la notation textuelle '<0.1' (ou valeurs seuils) dans la colonne Valeur de Sous-alimentation,Spécificité métier FAO — Impossibilité de conversion float directe ; décision de traitement : imputation à 0.05 ou 0.1


**Points de vigilance**

Périmètre géographique & Jointure : La jointure du dataset global devra explicitement documenter et justifier le sort des 27 pays de la table Sous-alimentation absents des trois bilans alimentaires et de la population (jointure interne assumée sur les 177 pays communs, avec la liste des 27 pays exclus fournie en annexe/documentation). La source officile FAOSTATs précise : **La comparabilité géographique est limitée en raison des différences de méthodes et de champ d'application, sauf pour les régions composées de pays homogènes**.

Identification des indicateurs : Le calcul des indicateurs (kcal/personne/jour, g/personne/jour, taux de sous-nutrition) devra obligatoirement s'appuyer sur le libellé complet de la colonne Élément et non sur la seule colonne Unité, afin d'éviter toute confusion entre les flux physiques (1000 t) et les ratios nutritionnels.

Traitement des valeurs négatives : Les 400 valeurs négatives associées à la « Variation de stock » doivent être conservées telles quelles (légitimité physique traduisant un déstockage net), tandis que les 281 valeurs négatives de « Disponibilité intérieure » devront faire l'objet d'un retraitement spécifique (recalcul ou mise à zéro).

Gestion des seuils de sous-alimentation : Les valeurs textuelles de type "<0.1" dans la table Sous-alimentation doivent être converties numériquement de façon explicite (ex: imputation à 0.05 ou 0.1 selon la règle de gestion choisie) avant toute agrégation ou calcul de ratio.

Isolation temporelle : La période d'analyse de référence devra être filtrée/isolée dès la première étape de préparation, avant toute opération de jointure entre les fichiers, pour éviter la multiplication indésirable de lignes.

## 3. Nettoyage, stratégies d'imputation et fusion des sources
**Objectif :** Nettoyer les sources de donénes suite au diagnostic préablement établis afin d'en ressortir un document global propre

In [10]:
#Calibrage de la mise en forme
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [11]:
#Rechargement datasets bruts
df_veg_raw = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_vegetaux.csv")
df_ani_raw = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_animaux.csv")
df_cer_raw = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_céréales.csv")  # Conservé/isolé si besoin d'analyse dédiée
df_pop_raw = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_population.csv")
df_sousalim_raw = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\Périodes_glissantes\fr_sousalimentation.csv")

**1. Nettoyage**
_______________________________
**A: Sélection temporelle**

*# Règle : La sélection temporelle a un double objectif -> limiter les doublons et traiter les valeurs manquantes uniquement pour la période nécessaire.*

le fichier sous-alimentation contient 1 632 doublons de clé dus aux périodes glissantes. On réduit à une période de référence permettant à la fois la suppression des doublons et la jointure avec les autres tables

In [12]:
PERIODE_REF = "2022-2024"
df_sousalim_ref = df_sousalim_raw[df_sousalim_raw["Année"] == PERIODE_REF].copy()
print(f"Lignes sur la période 2022-2024 : {len(df_sousalim_ref)}")
print(f"Type de la colonne Valeur : {df_sousalim_ref['Valeur'].dtype}")

Lignes sur la période 2022-2024 : 408
Type de la colonne Valeur : str


**B. Valeurs Manquantes**

*# Règle : La colonne 'Note' est vide à 100% sur les 5 fichiers (colonne résiduelle d'export), on décide donc de la supprimer*

In [13]:
datasets_raw = [
    df_veg_raw,
    df_ani_raw,
    df_cer_raw,
    df_pop_raw,
    df_sousalim_ref,
]
for df in datasets_raw:
    if "Note" in df.columns:
        df.drop(columns=["Note"], inplace=True)

*# Règle : Le type de la colonne valeur de sous-alimentation est actuellement en string, pour transformer les données, nous retenons une hypothèse conservatrice pour la valeur textuelle <0.1 million de la FAO en la fixant à sa borne supérieure 0.1 million (soit 100 000 personnes). En matière de sécurité alimentaire, privilégier la borne haute évite toute sous-estimation du risque de sous-nutrition dans les pays à faible prévalence.*

In [14]:
df_sousalim_ref["Valeur_clean"] = (
    df_sousalim_ref["Valeur"]
    .astype(str)
    .str.replace("<0.1", "0.1", regex=False)
    .str.replace("< 0.1", "0.1", regex=False)
    .str.replace("<2.5", "2.5", regex=False)
    .str.replace("< 2.5", "2.5", regex=False)
)
df_sousalim_ref["Valeur_num"] = pd.to_numeric(
    df_sousalim_ref["Valeur_clean"], errors="coerce"
)

*# Règle : La colonne 'Valeur' de la table Sous-alimentation (filtrée sur la période voulue) comptabilise 136 valeurs manquantes. Ayant chargé dans le dataset de 2023 un second indicateur, nous commençons par croiser les données afin d'obtenir une cohérence contextuelle pour le maximum de pays (Nombre vs Prévalence %) - On harmonise ainsi la population pour avoir les mêmes unités de mesure dans les calculs*

In [15]:
# Séparation des 2 indicateurs
mask_nombre = df_sousalim_ref["Produit"].str.contains(
    "Nombre", case=False, na=False
)
mask_prev = df_sousalim_ref["Produit"].str.contains(
    "Prévalence", case=False, na=False
)

df_nombre = df_sousalim_ref[mask_nombre][["Zone", "Valeur_num"]].rename(
    columns={"Valeur_num": "nb_millions_brut"}
)
df_prev = df_sousalim_ref[mask_prev][["Zone", "Valeur_num"]].rename(
    columns={"Valeur_num": "prevalence_pct_brut"}
)

# Rapprochement temporaire pour calcul
df_sousalim_croise = df_nombre.merge(df_prev, on="Zone", how="outer")

# Harmonisation de la Population : '1000 No' -> Nombre exact d'habitants
df_pop_clean = df_pop_raw.copy()
df_pop_clean["population"] = df_pop_clean["Valeur"] * 1_000

# Conversion de la Sous-alimentation en nombre réel d'habitants (Croisement)
# On fusionne temporairement la population pour pouvoir faire le calcul en habitants
df_sousalim_croise = df_sousalim_croise.merge(
    df_pop_clean[["Zone", "population"]], on="Zone", how="left"
)


def calculer_nb_sousalimentes_exact(row):
    # Priorité 1 : Si le nombre en millions est disponible (ex: 0.1 M -> 100 000 habs)
    if pd.notna(row["nb_millions_brut"]):
        return row["nb_millions_brut"] * 1_000_000
    # Priorité 2 : Calcul basé sur la prévalence (%) et la population
    elif pd.notna(row["prevalence_pct_brut"]) and pd.notna(row["population"]):
        return (row["prevalence_pct_brut"] / 100) * row["population"]
    # Sinon : On conserve NaN pour l'étape d'audit post-jointure
    else:
        return np.nan


df_sousalim_croise["nb_sousalimentes"] = df_sousalim_croise.apply(
    calculer_nb_sousalimentes_exact, axis=1
)

**C. Valeurs Négatives**

*# Règle de gestion sur des bilans : si il s'agit d'une variation de stock alors on considère la valeur comme légitime (déstockages) -> ON CONSERVE, s'il on mentionne une disponibilité intérieure inférieure à  0 alors on considère que c'est une anomalie déclarative -> ON IMPUTE À 0*

In [16]:
def nettoyer_valeurs_negatives(df):
    df_clean = df.copy()
    mask_disp_neg = (df_clean["Élément"] == "Disponibilité intérieure") & (
        df_clean["Valeur"] < 0
    )
    df_clean.loc[mask_disp_neg, "Valeur"] = 0
    return df_clean


df_veg_clean = nettoyer_valeurs_negatives(df_veg_raw)
df_ani_clean = nettoyer_valeurs_negatives(df_ani_raw)
df_cer_clean = nettoyer_valeurs_negatives(df_cer_raw)

df_veg_clean["Origine"] = "Végétale"
df_ani_clean["Origine"] = "Animale"

df_bilans = pd.concat([df_veg_clean, df_ani_clean], ignore_index=True)

**D. Consolidation et pivotage des bilans alimentaires par pays**

In [17]:
elements_souhaites = [
    "Disponibilité alimentaire (Kcal/personne/jour)",
    "Disponibilité de protéines en quantité (g/personne/jour)",
    "Disponibilité de matière grasse en quantité (g/personne/jour)",
    "Production",
    "Importations - quantité",
    "Exportations - quantité",
    "Disponibilité intérieure",
    "Pertes",
    "Variation de stock",
]

df_bilan_filtre = df_bilans[df_bilans["Élément"].isin(elements_souhaites)]

df_bilan_pivot = df_bilan_filtre.pivot_table(
    index="Zone", columns="Élément", values="Valeur", aggfunc="sum"
).reset_index()

df_bilan_pivot = df_bilan_pivot.rename(
    columns={
        "Disponibilité alimentaire (Kcal/personne/jour)": "dispo_calorique_totale_kcal_personne_jour",
        "Disponibilité de protéines en quantité (g/personne/jour)": "dispo_proteique_totale_g_personne_jour",
        "Disponibilité de matière grasse en quantité (g/personne/jour)": "dispo_gras_g_personne_jour",
        "Production": "production_ktonnes",
        "Importations - quantité": "importations_ktonnes",
        "Exportations - quantité": "exportations_ktonnes",
        "Disponibilité intérieure": "dispo_interieure_ktonnes",
        "Pertes": "pertes_ktonnes",
        "Variation de stock": "variation_stock_ktonnes",
    }
)

*Règle : La fonction `pivot_table` est utilisée pour restructurer le bilan alimentaire en passant d'un format long (multiples lignes par pays) à un format large (1 ligne unique par pays) : Le pivotement permet d'isoler et de sommer chaque unité séparément, évitant ainsi toute addition incohérente entre kilocalories, grammes de protéines et masses en tonnes.*


**2. Fusion des sources**

________________________________________

In [18]:
etape1 = df_bilan_pivot.merge(
    df_pop_clean[["Zone", "population"]],
    on="Zone",
    how="inner",
    indicator=True,
)

print("\n--- Étape 1 : Jointure INNER (Bilan + population) ---")
print(etape1["_merge"].value_counts())
etape1 = etape1.drop(columns="_merge")


dataset_global = etape1.merge(
    df_sousalim_croise[["Zone", "nb_sousalimentes"]],
    on="Zone",
    how="left",
    indicator=True,
)

print("\n--- Étape 2 : Jointure LEFT (+ sous-alimentation) ---")
print(dataset_global["_merge"].value_counts())

# Extraction de la liste des pays sans donnée de sous-nutrition
pays_sans_donnee_sousnutrition = sorted(
    dataset_global.loc[dataset_global["_merge"] == "left_only", "Zone"]
)

print(
    f"\n{len(pays_sans_donnee_sousnutrition)} pays sans donnée de sous-nutrition :\n"
)
for p in pays_sans_donnee_sousnutrition:
    print(" -", p)

dataset_global = dataset_global.drop(columns="_merge")
print(f"\nDataset global final : {len(dataset_global)} pays")


--- Étape 1 : Jointure INNER (Bilan + population) ---
_merge
both          177
left_only       0
right_only      0
Name: count, dtype: int64

--- Étape 2 : Jointure LEFT (+ sous-alimentation) ---
_merge
both          177
left_only       0
right_only      0
Name: count, dtype: int64

0 pays sans donnée de sous-nutrition :


Dataset global final : 177 pays


Jointure INNER JOIN (Kcal + Population) : Pour réaliser une analyse comparative cohérente, un pays doit impérativement posséder à la fois des données de bilan alimentaire ET un nombre d'habitants. L'INNER JOIN permet de créer le socle d'étude valide (les 177 pays) en éliminant automatiquement les agrégats régionaux ou encore ceux non présents dans les fichiers produits.

Jointure LEFT JOIN (+ Sous-alimentation) : Les données de sous-nutrition ne sont pas renseignées par la FAO pour tous les pays. Le LEFT JOIN garantit de conserver l'intégralité des 177 pays du socle d'analyse.

In [19]:
print(dataset_global.isna().sum())

Zone                                          0
dispo_calorique_totale_kcal_personne_jour     0
dispo_gras_g_personne_jour                    0
dispo_proteique_totale_g_personne_jour        0
dispo_interieure_ktonnes                      0
exportations_ktonnes                          0
importations_ktonnes                          0
pertes_ktonnes                                0
production_ktonnes                            0
variation_stock_ktonnes                       0
population                                    0
nb_sousalimentes                             20
dtype: int64


In [20]:
pays_sans_sousnut = dataset_global[dataset_global["nb_sousalimentes"].isna()]
print(f"--- LISTE EXHAUSTIVE DES {len(pays_sans_sousnut)} PAYS SANS DONNÉE DE SOUS-NUTRITION ---\n")
for i, pays in enumerate(sorted(pays_sans_sousnut["Zone"]), 1):
    print(f"{i:2d}. {pays}")

--- LISTE EXHAUSTIVE DES 20 PAYS SANS DONNÉE DE SOUS-NUTRITION ---

 1. Antigua-et-Barbuda
 2. Bahamas
 3. Bahreïn
 4. Bhoutan
 5. Chine - RAS de Hong-Kong
 6. Chine - RAS de Macao
 7. Grenade
 8. Lesotho
 9. Maldives
10. Micronésie (États fédérés de)
11. Naoero
12. Nicaragua
13. Qatar
14. Saint-Kitts-et-Nevis
15. Sainte-Lucie
16. Slovaquie
17. Tonga
18. Tuvalu
19. Yémen
20. Îles Marshall


*Règle : Nous conservons les 20 pays présentant une donnée manquante (NaN) en sous-nutrition dans le dataset final de 177 pays. Ces territoires disposent de données démographiques et alimentaires exploitables. Imputer ces NaN par 0, par la moyenne ou supprimer les lignes introduirait un biais statistique et priverait l'analyse de données réelles de disponibilité calorique.*

La liste met en avant 3 profils distincts : 
- Les micro-États et îles (Tuvalu, Nauru, Bahamas, Grenade, Maldives, etc.) : La FAO ne déploie pas toujours son modèle statistique de sous-nutrition sur les très petites populations.

- Les territoires à statut spécifique (Hong-Kong, Macao) : Nous avons décidé de ne pas regrouper en un seul pays en suivant les décisions géopolitiques de la FAO.

- Les zones avec rupture de remontée de données (Yémen, Nicaragua, Lesotho) : Manque de suivi statistique fiable sur la période.

**3. INDICATEURS CROISES**
___________________________________

**Objectif :** Le dataset actuel ne dispose pas de données pouvant expliquer les facteurs exogènes de la sous-nutrition par des variables géopolitiques, météréologiques ou sanitaires, la production d'indicateurs doit se focaliser sur les bilans alimentaires mondiaux et permettre une analyse détaillée 

*La disponibilité calorique moyenne par hab par jour (kcal/hab/j)* représente la quantité d'énergie (en kcal) quotidiennement accessible par habitant dans un pays pour couvrir ses besoins alimentaires.

*Taux de sous-nutrition (%)* : rapport entre le nombre de personnes sous-alimentées (nb_sousalimentes) et la population totale (population), exprimé en pourcentage. (Nécessite de s'assurer que les deux variables sont ramenées à la même unité, en nombre exact d'habitants).

*Taux de couverture des besoins caloriques (%)* : comparaison de la disponibilité calorique journalière par habitant (dispo_calorique_totale_kcal_personne_jour) par rapport aux apports nutritionnels recommandés par la FAO/OMS (2 500 kcal/personne/jour).

*Taux de couverture des besoins protéiques (%)* : comparaison de la disponibilité protéique journalière par habitant (dispo_proteique_totale_g_personne_jour) par rapport au besoin moyen recommandé (62 g/personne/jour).

*Taux d'autosuffisance (%)* : mesure la part de la disponibilité intérieure d'un pays directement couverte par sa propre production nationale (production_ktonnes / dispo_interieure_ktonnes).

*Taux de dépendance aux importations (%)* : mesure la part de la disponibilité intérieure qui repose sur les importations de denrées extérieures (importations_ktonnes / dispo_interieure_ktonnes).

*Taux de pertes post-récolte (%)* : évalue la proportion de la nourriture gâchée ou perdue au cours du stockage et du transport par rapport à la disponibilité intérieure (pertes_ktonnes / dispo_interieure_ktonnes).

*Population théoriquement nourrissable* : estimation du nombre d'individus que le pays pourrait nourrir à hauteur de 2 500 kcal/jour en mobilisant l'ensemble de sa disponibilité calorique globale ((dispo_calorique_totale_kcal_personne_jour * population) / 2500).

In [25]:
# --- CALCUL DES INDICATEURS CROISÉS AVANCÉS ---

# A. Disponibilité calorique totale du pays (en Kcal/jour pour toute la population)
dataset_global["dispo_calorique_totale_pays_kcal_jour"] = (
    dataset_global["dispo_calorique_totale_kcal_personne_jour"]
    * dataset_global["population"]
)

# B. Sous-nutrition et couverture des besoins (Normes OMS/FAO : 2500 Kcal, 62g Prot)
dataset_global["taux_sousnutrition_pct"] = (
    dataset_global["nb_sousalimentes"] / dataset_global["population"]
) * 100

dataset_global["taux_couverture_kcal_pct"] = (
    dataset_global["dispo_calorique_totale_kcal_personne_jour"] / 2500
) * 100

dataset_global["taux_couverture_prot_pct"] = (
    dataset_global["dispo_proteique_totale_g_personne_jour"] / 62
) * 100

# C. Souveraineté et vulnérabilité commerciale
dataset_global["taux_autosuffisance_pct"] = (
    dataset_global["production_ktonnes"]
    / dataset_global["dispo_interieure_ktonnes"]
) * 100

dataset_global["taux_dependance_import_pct"] = (
    dataset_global["importations_ktonnes"]
    / dataset_global["dispo_interieure_ktonnes"]
) * 100

# D. Pertes post-récolte et logistiques
dataset_global["taux_pertes_pct"] = (
    dataset_global["pertes_ktonnes"] / dataset_global["dispo_interieure_ktonnes"]
) * 100

# E. Capacité théorique d'alimentation (Population nourrissable)
dataset_global["pop_nourrissable_theorique"] = (
    dataset_global["dispo_calorique_totale_kcal_personne_jour"] * dataset_global["population"]
) / 2500


dataset_global.to_csv("dataset_global_FAO.csv", index=False)

In [29]:
# Sélection et renommage
cols_affichage = {
    "Zone": "Pays",
    "population": "Population (hab.)",
    "dispo_calorique_totale_kcal_personne_jour": "Dispo Kcal (kcal/hab/j)",
    "taux_sousnutrition_pct": "Sous-nutrition (%)",
    "taux_couverture_kcal_pct": "Couverture Kcal (%)",
    "taux_couverture_prot_pct": "Couverture Prot. (%)",
    "taux_autosuffisance_pct": "Autosuffisance (%)",
    "taux_dependance_import_pct": "Dépendance Import. (%)",
    "taux_pertes_pct": "Pertes (%)",
    "pop_nourrissable_theorique": "Pop. nourrissable (hab.)",
}

# 2. On sélectionne et on renomme directement
dataset_global[list(cols_affichage)].rename(columns=cols_affichage).head(10)
df_style = dataset_global[list(cols_affichage.keys())].rename(
    columns=cols_affichage
)

# Application du formatage visuel sur les 10 premières lignes
df_style.head(10).style.format({
    "Population (hab.)": "{:,.0f}",
    "Dispo Kcal (kcal/hab/j)" : "{:,.0f} Kcal/hab/j",
    "Sous-nutrition (%)": "{:.2f} %",
    "Couverture Kcal (%)": "{:.1f} %",
    "Couverture Prot. (%)": "{:.1f} %",
    "Autosuffisance (%)": "{:.1f} %",
    "Dépendance Import. (%)": "{:.1f} %",
    "Pertes (%)": "{:.1f} %",
    "Pop. nourrissable (hab.)": "{:,.0f}",
})

,Pays,Population (hab.),Dispo Kcal (kcal/hab/j),Sous-nutrition (%),Couverture Kcal (%),Couverture Prot. (%),Autosuffisance (%),Dépendance Import. (%),Pertes (%),Pop. nourrissable (hab.)
0,Afghanistan,"41,454,760","2,315 Kcal/hab/j",29.91 %,92.6 %,100.9 %,72.8 %,35.8 %,7.0 %,"38,381,138"
1,Afrique du Sud,"63,212,380","2,694 Kcal/hab/j",10.60 %,107.8 %,127.7 %,107.1 %,16.7 %,3.0 %,"68,116,902"
2,Albanie,"2,811,660","3,238 Kcal/hab/j",7.11 %,129.5 %,183.8 %,82.5 %,24.5 %,6.7 %,"3,641,437"
3,Algérie,"46,164,220","3,396 Kcal/hab/j",2.50 %,135.8 %,150.9 %,59.2 %,38.9 %,7.4 %,"62,709,107"
4,Allemagne,"84,548,230","3,535 Kcal/hab/j",2.50 %,141.4 %,176.1 %,83.2 %,49.1 %,2.6 %,"119,534,626"
5,Angola,"36,749,910","2,399 Kcal/hab/j",18.50 %,96.0 %,76.5 %,90.7 %,7.6 %,8.2 %,"35,269,918"
6,Antigua-et-Barbuda,"93,320","2,544 Kcal/hab/j",nan %,101.8 %,158.2 %,15.0 %,90.3 %,0.0 %,"94,972"
7,Arabie saoudite,"33,264,290","3,660 Kcal/hab/j",2.50 %,146.4 %,167.4 %,39.0 %,76.5 %,2.3 %,"48,696,925"
8,Argentine,"45,538,400","3,360 Kcal/hab/j",3.07 %,134.4 %,196.0 %,126.3 %,9.5 %,2.7 %,"61,211,807"
9,Arménie,"2,943,390","3,129 Kcal/hab/j",2.50 %,125.2 %,166.4 %,73.4 %,36.8 %,4.6 %,"3,684,265"


**4. Contrôle de plausibilité**

Avant de considérer les indicateurs comme fiables, on vérifie qu'ils restent dans des plages plausibles.

In [30]:
# Le taux de sous-nutrition ne peut pas dépasser 100 % de la population
mask_sousnut_aberrant = dataset_global["taux_sousnutrition_pct"] > 100

nb_anomalies_sousnut = mask_sousnut_aberrant.sum()
if nb_anomalies_sousnut > 0:
    print(
        f"{nb_anomalies_sousnut} pays avec un taux de sous-nutrition > 100% : plafonné(s) à 100 %."
    )
    dataset_global.loc[mask_sousnut_aberrant, "taux_sousnutrition_pct"] = 100.0
else:
    print("Taux de sous-nutrition : aucune valeur > 100 %.")

Taux de sous-nutrition : aucune valeur > 100 %.


In [31]:
# Détection de valeurs négatives impossibles 

cols_positives = [
    "population",
    "dispo_calorique_totale_kcal_personne_jour",
    "dispo_proteique_totale_g_personne_jour",
    "taux_sousnutrition_pct",
]

for col in cols_positives:
    valeurs_negatives = (dataset_global[col] < 0).sum()
    if valeurs_negatives > 0:
        print(f"Alerte : {valeurs_negatives} valeur(s) négative(s) dans '{col}' !")
    else:
        print(f"Colonne '{col}' : aucune valeur négative.")

Colonne 'population' : aucune valeur négative.
Colonne 'dispo_calorique_totale_kcal_personne_jour' : aucune valeur négative.
Colonne 'dispo_proteique_totale_g_personne_jour' : aucune valeur négative.
Colonne 'taux_sousnutrition_pct' : aucune valeur négative.


In [32]:
# Récapitulatif des valeurs manquantes 

print("\n Nombre de valeurs manquantes (NaN) par indicateur :")
nan_summary = dataset_global.isna().sum()
print(nan_summary[nan_summary > 0])


 Nombre de valeurs manquantes (NaN) par indicateur :
nb_sousalimentes          20
taux_sousnutrition_pct    20
dtype: int64


# Etape 2 : Analyse exploratoire 

# Etape 3 : Modelisation : régression 

# Etape 4 : Clustering et restitution